# Phase 1: Inspect `twcs.csv` — Schema & Brand Volume

### Objective
1. **Download & Verify**: Set up Kaggle API authentication in Colab/local, download `twcs.csv`, verify file integrity, file size (~530 MB), and count total rows (~2.81M) without running out of memory.
2. **Schema Inspection**: Inspect each column (`tweet_id`, `author_id`, `inbound`, `created_at`, `text`, `response_tweet_id`, `in_response_to_tweet_id`) and understand their precise role in conversation graph reconstruction.
3. **Brand Ranking & Exploration**: Scan chunked records to rank the top brand `author_id`s by volume to inform our brand selection.

---
### Why this is critical for the assignment
- TWCS is **not** a clean list of dialogs. It is an unrolled relational table of graph nodes with parent (`in_response_to_tweet_id`) and children (`response_tweet_id`) references.
- Attempting `pd.read_csv('twcs.csv')` carelessly can consume gigabytes of RAM or crash free Colab instances when performing joins.
- Selecting the right brand is a foundational architectural decision: brands differ drastically in support complexity (e.g., tech troubleshooting vs. delivery tracking vs. airline rebooking).

In [ ]:
# Setup environment & Kaggle download (Run in Colab if needed)
!mkdir -p data/raw data/processed data/evaluation reports configs

# If running in Colab and using Kaggle credentials:
# from google.colab import files
# files.upload() # upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d thoughtvector/customer-support-on-twitter -p data/raw/ --unzip

In [ ]:
import os
import sys
import pandas as pd
from collections import Counter

data_path = "data/raw/twcs.csv"

# Step 1: Verify file exists and print size
if not os.path.exists(data_path):
    raise FileNotFoundError(f"{data_path} not found. Please download from Kaggle first.")

size_mb = os.path.getsize(data_path) / (1024 * 1024)
print(f"File found: {data_path}")
print(f"File size: {size_mb:.2f} MB")

# Stream line count to prevent memory spike
line_count = 0
with open(data_path, 'r', encoding='utf-8', errors='replace') as f:
    for _ in f:
        line_count += 1

print(f"Total lines (incl. header): {line_count:,}")
print(f"Total tweet rows: {line_count - 1:,}")

In [ ]:
# Step 2: Load safe sample and inspect schema & data types
df_sample = pd.read_csv(data_path, nrows=10)
display(df_sample.head())
print("\n--- Data Types & Non-Null Counts in Sample ---")
print(df_sample.info())

### Column Explanations for Conversation Reconstruction

| Column | Type | Role in Thread Reconstruction |
|---|---|---|
| `tweet_id` | Int64 | Unique tweet identifier (node key in conversation DAG). |
| `author_id` | String | Anonymized user ID (e.g. `115712`) or public brand handle (e.g. `AppleSupport`, `AmazonHelp`). |
| `inbound` | Bool | Direction: `True` = customer to brand; `False` = brand to customer. |
| `created_at` | String | Datetime string. Critical to sort conversation turns chronologically. |
| `text` | String | Tweet text with @mentions and masked URLs (`https://t.co/...`). |
| `response_tweet_id` | String (CSV) | Forward edge(s): child tweet IDs responding to this tweet. Can be multiple if bifurcated. |
| `in_response_to_tweet_id` | Float64 / Int | Backward edge: parent tweet ID. **If null/NaN, this tweet is the ROOT (conversation starter)!** |

In [ ]:
# Step 3: Chunked scan for Top Brand author_ids by reply volume
# Brands are defined as author_ids with inbound == False
brand_counts = Counter()
chunk_size = 250000

print("Scanning twcs.csv in chunks to count brand activity...")
for i, chunk in enumerate(pd.read_csv(data_path, usecols=["author_id", "inbound"], chunksize=chunk_size)):
    brand_replies = chunk[chunk["inbound"] == False]["author_id"]
    brand_counts.update(brand_replies.value_counts().to_dict())
    print(f"Processed chunk {i+1} ({chunk_size * (i+1):,} rows processed)...")

top_brands = pd.DataFrame(brand_counts.most_common(20), columns=["Brand Handle", "Outbound Tweet Count"])
display(top_brands)